In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mitigazione degli errori con reti tensoriali (TEM): una Qiskit Function di Algorithmiq
> **Note:** Le Qiskit Functions sono una funzionalità sperimentale disponibile solo per gli utenti del piano IBM Quantum&reg; Premium, del piano Flex e del piano On-Prem (tramite IBM Quantum Platform API). Sono in stato di anteprima e soggette a modifiche.


<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Panoramica
Il metodo TEM (Tensor-network Error Mitigation) di Algorithmiq è un algoritmo
ibrido quantistico-classico progettato per eseguire la mitigazione del rumore
interamente nella fase di post-elaborazione classica. Con TEM, puoi calcolare i
valori attesi degli osservabili mitigando gli inevitabili errori indotti dal
rumore che si verificano sull'hardware quantistico, con maggiore precisione ed
efficienza dei costi: un'opzione molto attraente sia per i ricercatori nel campo
quantistico sia per i professionisti del settore industriale.

Il metodo consiste nel costruire una rete tensoriale che rappresenta l'inverso
del canale di rumore globale che agisce sullo stato del processore quantistico,
per poi applicare la mappa agli esiti di misurazione informativamente completi
acquisiti dallo stato rumoroso, ottenendo stimatori non distorti per gli
osservabili.

Come vantaggio, TEM sfrutta misurazioni informativamente complete per dare
accesso a un vasto insieme di valori attesi mitigati degli osservabili e ha un
overhead di campionamento ottimale sull'hardware quantistico, come descritto in
Filippov et al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), e
Filippov et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542).
L'overhead di misurazione si riferisce al numero di misurazioni aggiuntive
necessarie per eseguire una mitigazione degli errori efficiente, un fattore
critico per la fattibilità dei calcoli quantistici. Pertanto, TEM ha il
potenziale per abilitare il vantaggio quantistico in scenari complessi, come
applicazioni nei campi del caos quantistico, della fisica a molti corpi, della
dinamica di Hubbard e delle simulazioni di chimica di piccole molecole.

Le principali caratteristiche e i vantaggi di TEM possono essere riassunti come
segue:

1.  **Overhead di misurazione ottimale**: TEM è ottimale rispetto ai
[limiti teorici](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
nel senso che nessun metodo può raggiungere un overhead di misurazione inferiore.
In altre parole, TEM richiede il numero minimo di misurazioni aggiuntive per
eseguire la mitigazione degli errori. Questo significa che TEM utilizza il tempo
di esecuzione quantistica minimo.
2.  **Convenienza economica**: poiché TEM gestisce la mitigazione del rumore
interamente nella fase di post-elaborazione, non è necessario aggiungere circuiti
extra al computer quantistico. Ciò non solo rende il calcolo meno costoso, ma
riduce anche il rischio di introdurre errori aggiuntivi dovuti alle imperfezioni
dei dispositivi quantistici.
3.  **Stima di osservabili multipli**: grazie alle misurazioni informativamente
complete, TEM stima in modo efficiente più osservabili con gli stessi dati di
misurazione del computer quantistico.
4.  **Mitigazione degli errori di lettura**: la Qiskit Function TEM include anche
un [metodo proprietario di mitigazione degli errori di lettura](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
in grado di ridurre significativamente gli errori di readout dopo una breve
sessione di calibrazione.
5.  **Precisione**: TEM migliora significativamente la precisione e l'affidabilità
delle simulazioni quantistiche digitali, rendendo gli algoritmi quantistici più
accurati e affidabili.
## Descrizione
La funzione TEM ti permette di ottenere valori attesi con mitigazione degli errori
per più osservabili su un circuito quantistico con un overhead di campionamento
minimo. Il circuito viene misurato con una misura a operatore a valore positivo
informativamente completa (IC-POVM), e gli esiti di misurazione raccolti vengono
elaborati su un computer classico. Questa misurazione viene utilizzata per
applicare i metodi di reti tensoriali e costruire una mappa di inversione del
rumore. La funzione applica una mappa che inverte completamente l'intero circuito
rumoroso usando reti tensoriali per rappresentare i layer rumorosi.

![Schema TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Stima con mitigazione degli errori di un osservabile O tramite post-elaborazione degli esiti di misurazione del processore quantistico rumoroso. U e N denotano un'operazione quantistica ideale e la mappa di rumore associata, che può essere in generale non locale (ed estesa alle caselle grigie). D rappresenta un tensore di operatori duali agli effetti nella misurazione IC. Il modulo di mitigazione del rumore M è una rete tensoriale che viene contratta in modo efficiente dall'interno verso l'esterno. La prima iterazione della contrazione è rappresentata dalla linea viola tratteggiata, la seconda dalla linea a tratti e la terza dalla linea continua.")

Una volta inviati i circuiti alla funzione, questi vengono traspilati e
ottimizzati per minimizzare il numero di layer con gate a due qubit (i gate più
rumorosi sui dispositivi quantistici). Il rumore che agisce sui layer viene
appreso tramite
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
usando un modello di rumore sparse Pauli-Lindblad come descritto in E. van den
Berg, Z. Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Il modello di rumore è una descrizione accurata del rumore sul dispositivo, in
grado di catturare caratteristiche sottili, tra cui il crosstalk tra qubit.
Tuttavia, il rumore sui dispositivi può fluttuare e derivare, e il rumore appreso
potrebbe non essere preciso nel momento in cui viene eseguita la stima. Questo
potrebbe portare a risultati imprecisi.
## Inizia
Autenticati usando la tua [chiave API di IBM Quantum Platform](http://quantum.cloud.ibm.com/), e seleziona la funzione TEM come segue. (Questo frammento presuppone che tu abbia già [salvato il tuo account](/guides/functions#install-qiskit-functions-catalog-client) nell'ambiente locale.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Esempio
Il seguente frammento mostra un esempio in cui TEM viene utilizzato per calcolare i valori attesi di un osservabile dato un semplice circuito quantistico.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Usa le API di Qiskit Serverless per controllare lo stato del tuo workload Qiskit Function:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Puoi ottenere i risultati nel modo seguente:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** Il valore atteso per il circuito senza rumore per l'operatore dato dovrebbe essere circa `0.18409094298943401`.
## Input
**Parametri**

Nome | Tipo | Descrizione | Obbligatorio | Predefinito | Esempio
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Un iterabile di oggetti simili a PUB (primitive unified bloc), come tuple `(circuit, observables)` o `(circuit, observables, parameters, precision)`. Vedi [Panoramica dei PUB](/guides/primitive-input-output#overview-of-pubs) per ulteriori informazioni. Se viene passato un circuito non-ISA, verrà traspilato con impostazioni ottimali. Se viene passato un circuito ISA, non verrà traspilato; in questo caso, l'osservabile deve essere definito sull'intero QPU. | Sì | N/A | (circuit, observables)
`backend_name` | str | Nome del backend su cui eseguire la query. | No | Se non fornito, verrà utilizzato il backend meno occupato. | "ibm_fez"
`options` | dict | Opzioni di input. Vedi la sezione `Options` per maggiori dettagli. | No | Vedi la sezione `Options` per i valori predefiniti. | {"max_bond_dimension": 100\}

> **Caution:** TEM presenta attualmente le seguenti limitazioni:
> 
>   - I circuiti parametrizzati non sono supportati. L'argomento parameters deve essere impostato su `None` se la precisione è specificata. Questa limitazione verrà rimossa nelle versioni future.
>   - Sono supportati solo circuiti senza cicli. Questa limitazione verrà rimossa nelle versioni future.
>   - I gate non unitari, come reset, measure e tutte le forme di control flow, non sono supportati. Il supporto per reset verrà aggiunto nelle prossime versioni.
### Opzioni
Un dizionario contenente le opzioni avanzate per TEM. Il dizionario può contenere le chiavi indicate nella tabella seguente. Se alcune opzioni non vengono fornite, verrà usato il valore predefinito indicato nella tabella. I valori predefiniti sono adatti all'uso tipico di TEM.

Nome | Valori | Descrizione | Predefinito
-- | -- | -- | --
`tem_max_bond_dimension` | int | La dimensione massima del bond da usare per le reti tensoriali. | 500 |
`tem_compression_cutoff` | float | Il valore di cutoff da usare per le reti tensoriali. | 1e-16
`compute_shadows_bias_from_observable` | bool | Un flag booleano che indica se il bias per il protocollo di misurazione a classical shadows deve essere adattato all'osservabile del PUB oppure no. Se False, verrà usato il protocollo classical shadows standard (uguale probabilità di misurare Z, X, Y). | False
`shadows_bias` | np.ndarray | Il bias da usare per il protocollo di misurazione a classical shadows randomizzato, un array 1d o 2d di dimensione 3 o forma (num_qubits, 3) rispettivamente. L'ordine è ZXY. | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int o `None` | Il tempo massimo di esecuzione sul QPU in secondi. Se il tempo di esecuzione supera questo valore, il job verrà annullato. Se `None`, si applica un limite predefinito impostato da Qiskit Runtime. | `None`
`num_randomizations` | int | Il numero di randomizzazioni da usare per l'apprendimento del rumore e il gate twirling. | 32
`max_layers_to_learn` | int | Il numero massimo di layer unici da apprendere. | 4
`mitigate_readout_error` | bool | Un flag booleano che indica se eseguire o meno la mitigazione degli errori di readout. | True
`num_readout_calibration_shots` | int | Il numero di shot da usare per la mitigazione degli errori di readout. | 10000
`default_precision` | float | La precisione predefinita da usare per i PUB per i quali la precisione non è specificata. | 0.02
`seed` | int o `None` | Imposta il seed del generatore di numeri casuali per la riproducibilità. Se `None`, il seed non viene impostato. | None
## Output
Un [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) di Qiskit contenente il risultato mitigato da TEM. Il risultato per ciascun PUB viene restituito come [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) contenente i seguenti campi:

Nome | Tipo | Descrizione
-- | -- | --
data | DataBin | Un [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) di Qiskit contenente l'osservabile mitigato da TEM e il suo errore standard. Il DataBin ha i seguenti campi: <ul><li>`evs`: il valore dell'osservabile mitigato da TEM.</li><li>`stds`: l'errore standard dell'osservabile mitigato da TEM.</li></ul>
metadata | dict | Un dizionario contenente risultati aggiuntivi. Il dizionario contiene le seguenti chiavi: <ul><li>`"evs_non_mitigated"`: il valore dell'osservabile senza mitigazione degli errori.</li><li>`"stds_non_mitigated"`: l'errore standard del risultato senza mitigazione degli errori.</li><li>`"evs_mitigated_no_readout_mitigation"`: il valore dell'osservabile con mitigazione degli errori ma senza mitigazione degli errori di readout.</li><li>`"stds_mitigated_no_readout_mitigation"`: l'errore standard del risultato con mitigazione degli errori ma senza mitigazione degli errori di readout.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: il valore dell'osservabile senza mitigazione degli errori ma con mitigazione degli errori di readout.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: l'errore standard del risultato senza mitigazione degli errori ma con mitigazione degli errori di readout.</li></ul>
## Recupero dei messaggi di errore
Se lo stato del tuo workload è ERROR, usa job.result() per recuperare il messaggio di errore nel modo seguente: